# Storage audit and safe step-10 recovery

**Start in a CPU runtime. No further GPU training until the audit and quota preflight pass.**

The legacy runner saved two full smoke checkpoints, removed them through the Drive mount, and then forced checkpoint-10 when its reserve fired. The user confirmed roughly 8 GB of generated smoke artifacts in Trash, which still counts against quota. This notebook never empties Trash, never deletes checkpoint-10, and requires an exact reviewed confirmation for every permanent deletion.

All destructive and training actions default OFF. Preserve the original `source_commit.json`, `run_config.json`, prepared data and checkpoint-10. This profile freezes its execution source separately. New full checkpoints use local staging, lossless compression/XOR against an immutable base, actual Drive API quota, validated replacement publication and explicit-confirmation retention. Full optimizer/scheduler/RNG state is preserved. Full-size compression and live API recovery remain unverified until measured here.

In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
RUN_DIR = Path("/content/drive/MyDrive/ArabicNewsSummarizer/full-v2-fp32-final-01")
CODE_REF = "codex/storage-safe-recovery"  # exact PR commit also works; use main after merge
LOCAL_ROOT = Path("/content/arabic-storage") / hashlib.sha256(str(RUN_DIR).encode()).hexdigest()[:16]
ENABLE_TRAINING = False
SEED_LEGACY_WITHOUT_GPU = False
APPLY_SMOKE_TRASH_DELETION = False
APPLY_MANAGED_RETENTION = False
print("Existing run:", RUN_DIR, "Local staging:", LOCAL_ROOT)


## Load the storage correction without rewriting the original source lock

Use a fresh kernel. The original run's source pin stays unchanged; the storage profile has a separate execution pin under `storage-v1/`. A previous partial storage run resumes at its frozen storage-code SHA. No model weights are downloaded by this bootstrap.

In [ ]:
if any(name == "arabic_summarizer" or name.startswith("arabic_summarizer.") for name in sys.modules):
    raise RuntimeError("Restart the runtime to load a different project source safely.")
repo = Path("/content/ArabicNewsStorageCode")
url = "https://github.com/nada-ali-13/Arabic-news-summarizer.git"
if not (repo / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", url, str(repo)], check=True)
else:
    if subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain"], text=True).strip():
        raise RuntimeError("Preserve local source edits and use a fresh runtime.")
lock = RUN_DIR / "storage-v1/runtime_source_commit.json"
ref = json.loads(lock.read_text())["commit"] if lock.exists() else CODE_REF
subprocess.run(["git", "-C", str(repo), "fetch", "origin", ref], check=True)
commit = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "FETCH_HEAD"], text=True).strip()
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", commit], check=True)
if not (repo / "src/arabic_summarizer/recovery.py").exists():
    raise RuntimeError("Source lacks the storage fix; select the storage PR branch/commit, not old main.")
if not lock.exists():
    lock.parent.mkdir(parents=True, exist_ok=True)
    tmp = lock.with_suffix(".tmp")
    tmp.write_text(json.dumps({"commit": commit, "legacy_source_lock": str(RUN_DIR / "source_commit.json")}))
    tmp.replace(lock)
print("Storage execution commit:", commit)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-storage.txt")], check=True)
sys.path.insert(0, str(repo / "src"))


## Read-only account audit

Authorize normal Google access using the same account as the Drive mount. This stage reads quota and owned file metadata, including Trash and ancestor IDs. It downloads no personal file contents and performs no deletion. The report is private in local Colab storage. `other_*` means outside the supplied folder lineage, not pre-existing user data. Missing ancestry is never permission to bulk-delete files by name.

In [ ]:
import os

from huggingface_hub import constants as hub_constants

from arabic_summarizer.drive_storage import DriveBackend, account_audit, colab_service, find_folder
from arabic_summarizer.io import atomic_json
from arabic_summarizer.recovery import StorageRecovery, load_run_config
from arabic_summarizer.storage import ManagedStore, StorageBlocked, StoragePolicy
from arabic_summarizer.trash_cleanup import apply_smoke_cleanup, preview_smoke_cleanup

service = colab_service()
project_id = find_folder(service, RUN_DIR.relative_to("/content/drive/MyDrive").parts)
audit = account_audit(service, project_id)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
atomic_json(LOCAL_ROOT / "account-audit.json", audit)
print(json.dumps({"quota": audit["quota"], "totals": audit["totals_bytes"]}, indent=2))
for row in audit["largest_files"][:30]:
    print(row)
print("Every total is in bytes. GB = bytes / 1e9; GiB = bytes / 2**30.")


print("HF model cache:", hub_constants.HF_HUB_CACHE)
for name in ("HF_HOME", "HF_HUB_CACHE", "HF_DATASETS_CACHE", "TRANSFORMERS_CACHE"):
    if os.environ.get(name):
        print(name, os.environ[name])  # Only known non-secret cache paths; never credentials.


## Preview only obsolete smoke Trash entries; preserve checkpoint-10

The preview validates every file in checkpoint-10 with its SHA-256 manifest. It offers only trashed files with proven project → experiments → smoke → checkpoint-1/2 ancestry and an expected generated filename. If the lineage is missing, it refuses to infer ownership from names. Review the report and use Drive's own original-location metadata for any manual fallback; never select the whole Trash.

Set `APPLY_SMOKE_TRASH_DELETION=True` only after reviewing this exact list. The deletion cell then requires typing its exact confirmation phrase. Broader Drive delete authorization for old mount-created files is requested only after that confirmation. No deletion occurs by default.

In [ ]:
config = load_run_config(RUN_DIR)
protected = RUN_DIR / "experiments/arat5-5e5/checkpoint-10"
smoke_plan = preview_smoke_cleanup(audit, protected, config.fingerprint)
atomic_json(LOCAL_ROOT / "smoke-cleanup-plan.json", smoke_plan)
print(json.dumps(smoke_plan, indent=2))


In [ ]:
if APPLY_SMOKE_TRASH_DELETION:
    answer = input("Review the plan above. Type its exact confirmation phrase: ")
    if answer != smoke_plan["confirmation"]:
        raise RuntimeError("Confirmation did not match; nothing deleted.")
    delete_service = colab_service(allow_legacy_delete=True)
    receipt = apply_smoke_cleanup(delete_service, smoke_plan, answer)
    atomic_json(LOCAL_ROOT / "smoke-cleanup-receipt.json", receipt)
    atomic_json(RUN_DIR / "storage-v1/legacy-smoke-cleanup-receipt.json", receipt)
    print(json.dumps(receipt, indent=2))
else:
    print("Read-only: no Trash deletion requested.")


## Actual-quota plan and managed cleanup preview

Requery account quota after cleanup. Do not assume that a delete immediately updates quota. The immutable original checkpoint-10 remains on Drive. Lossless compression is measured locally before training and may not be sufficient; no ratio is promised. The preflight reserves replacement space plus overhead and a 1 GB margin, rather than merely lowering the old reserve.

Only newly created managed snapshots can be retired by managed retention. Their replacement and remote index must pass validation first, then you must confirm the exact obsolete ID list. The next training attempt pauses if an approval is pending. This explicit-review policy can require intervention at checkpoint intervals; it does not silently accumulate Trash.
The report distinguishes current measured replacement size from the worst-case full lifecycle. Training requires `full_plan_fits=True`; it does not assume step-10 compression will hold through 1,200 steps. With the reported protected original and full-size archives, the worst-case peak is about 16.64 decimal GB, which can exceed even a 15 GiB account. CPU adoption/roundtrip may still be verified, but further GPU training remains blocked until capacity or a separately reviewed backup/offload resolves the shortage.


In [ ]:

run_id = hashlib.sha256(str(RUN_DIR.resolve()).encode()).hexdigest()[:20]
backend = DriveBackend(service, run_id)
policy = StoragePolicy(**json.loads((repo / "configs/storage_colab_15gb.json").read_text()))
controller = StorageRecovery(RUN_DIR, backend, LOCAL_ROOT, policy)
print(json.dumps(controller.audit_plan(), indent=2))
store = ManagedStore(backend, LOCAL_ROOT, config.fingerprint, policy)
managed_plan = store.cleanup_plan()
print(json.dumps(managed_plan, indent=2))
if APPLY_MANAGED_RETENTION and managed_plan["obsolete"]:
    answer = input("Review the obsolete managed IDs. Type the exact confirmation phrase: ")
    store.cleanup(confirmation=answer)
    print("Requested managed cleanup completed; quota will be rechecked before another write.")
else:
    print("No managed deletion performed.")


## Optional no-GPU checkpoint adoption

`SEED_LEGACY_WITHOUT_GPU=True` copies the valid step-10 state into a lossless managed archive and verifies it remotely, then downloads/decodes it and checks every restored file hash. Inspect `storage-v1/recovery-verification.json` before the first T4 resume. It does not train, alter the original checkpoint, or delete anything. It still requires enough quota for the protected original and a validated replacement. Keep its receipt; new archives use the original model file as their immutable delta base.

In [ ]:
if SEED_LEGACY_WITHOUT_GPU:
    try:
        adopted = controller.seed_legacy()
        print(json.dumps(adopted, indent=2))
    except StorageBlocked as exc:
        print("BLOCKED before GPU training:", exc)
else:
    print("No adoption/upload requested.")


## Resume only when the quota plan passes

After the audit/cleanup, start a fresh T4 runtime with the same RUN_DIR, LOCAL_ROOT pattern and execution source pin. Leave cleanup flags false unless an explicit known batch needs approval. Set `ENABLE_TRAINING=True`. No original source/config lock is rewritten. Smoke/baseline/completed experiments are reused; checkpoint-10 resumes optimizer/scheduler/RNG state. Do not point this at a new empty directory when recovering.

At the moment of resuming, the live quota is checked before model allocation, and again before every upload. If compression/active+Trash usage makes the plan infeasible, it stops safely. A complete checkpoint at an experiment's final step is finalized without taking an extra optimizer step after a cleanup pause. Full-model T4 recovery with this storage profile and live Drive API operations have not been executed by Codex; local real-Trainer regressions cover the transfer/recovery protocol.

In [ ]:
if ENABLE_TRAINING:
    import torch

    from arabic_summarizer.training import run
    if not torch.cuda.is_available():
        raise RuntimeError("Audit is CPU-safe; select a fresh T4 runtime only when ready to resume.")
    try:
        result = run(controller.config, RUN_DIR, storage=controller)
        print(json.dumps(result, ensure_ascii=False, indent=2))
    except StorageBlocked as exc:
        print("Paused safely:", exc)
else:
    print("Training disabled. Audit and review quota/cleanup first.")
